# NLLB quz↔spa LoRA fine-tune — one epoch per Kaggle session

Each Kaggle session trains **one full epoch** over the `quz` parallel corpus, starting from
the previous session's pushed model (so epoch 2 continues from epoch 1's result, etc.), and
pushes to a **single Hub model repo** (`gmezan/nllb-quechua-spanish`), tagging each epoch's
commit (`epoch-1`, `epoch-2`, `epoch-3`) so every epoch stays individually retrievable even
though they all share one repo id -- `main` always holds the most recently pushed epoch, and
each tag pins that specific epoch's weights forever, regardless of what gets pushed later.

**Before running:** Settings → Accelerator → GPU, Settings → Internet → On, and add an
`HF_TOKEN` secret (Add-ons → Secrets).

**Between sessions:** just change `EPOCH_NUM` in the next cell (1, then 2, then 3) and re-run
the whole notebook — everything else derives from it automatically.

In [ ]:
# ✏️ EDIT THIS each session: 1 for the first session, 2 for the second, 3 for the third.
EPOCH_NUM = 1

HF_USERNAME = "gmezan"
BASE_MODEL_SEED = "facebook/nllb-200-distilled-600M"

# One repo for every epoch -- matches the default --model/--hub-repo-id everywhere else in
# the project, so nothing downstream (evaluation/, app/) needs to know epoch numbers exist.
HUB_REPO_ID = f"{HF_USERNAME}/nllb-quechua-spanish"
TAG = f"epoch-{EPOCH_NUM}"

# Epoch 1 starts from the untuned base model; epoch N (N>1) continues from whatever's on
# HUB_REPO_ID's main branch -- which, by the time session N starts, is epoch N-1's pushed
# result (each session pushes to main before tagging it), so this always picks up correctly
# without needing to reference the previous epoch's tag explicitly.
BASE_MODEL = BASE_MODEL_SEED if EPOCH_NUM == 1 else HUB_REPO_ID

print(f"Epoch {EPOCH_NUM}: base model = {BASE_MODEL}")
print(f"Epoch {EPOCH_NUM}: pushing to  = {HUB_REPO_ID}  (tag: {TAG})")

In [ ]:
import os
import sys
import runpy

REPO_DIR = "/kaggle/working/speech-to-speech-pipeline"
NLP_DIR = f"{REPO_DIR}/nlp"
EVAL_DIR = f"{REPO_DIR}/evaluation"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/gmezan/speech-to-speech-pipeline.git {REPO_DIR}
else:
    print("Repo already present, skipping clone.")

In [ ]:
# Kaggle's base image already has a GPU-matched torch install -- don't reinstall it,
# just add the extra deps train.py/eval_mt.py need.
!pip install -q accelerate peft sentencepiece python-dotenv sacrebleu

# Kaggle ships a torchao version too old for the peft version pip just installed;
# peft only *checks* for torchao (this project never uses it), so removing it entirely
# makes peft treat it as "not available" and skip that check.
!pip uninstall -y torchao

In [ ]:
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")

for env_dir in (NLP_DIR, EVAL_DIR):
    with open(f"{env_dir}/.env", "w") as f:
        f.write(f"HF_TOKEN={hf_token}\nHF_USERNAME={HF_USERNAME}\n")

print("Wrote .env files for nlp/ and evaluation/")

In [ ]:
import torch

print(f"GPUs visible: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
def run_cli(script_path, argv):
    """Run a CLI script in-process, as `python script_path argv...` would, but inside
    this notebook's own kernel instead of a subprocess -- so tqdm/Trainer correctly
    detect the Jupyter/IPython environment and render a live progress *widget* instead
    of spamming a new printed line per update (which is what happens through a `!python`
    subprocess, since its stdout isn't a real terminal and tqdm can't overwrite in place).

    nlp/translation/ and evaluation/ each have their own unrelated `common.py` -- once
    one is imported as the top-level module `common`, Python's import cache would wrongly
    hand back that *same* module to the other script's `from common import ...` without
    this. So this scopes sys.path to just the script's own directory and clears the
    cached `common` module before every run, keeping each script's import independent.
    """
    script_dir = os.path.dirname(script_path)
    old_argv, old_path, old_cwd = sys.argv, sys.path[:], os.getcwd()
    sys.modules.pop("common", None)
    sys.argv = [script_path, *argv]
    sys.path.insert(0, script_dir)
    os.chdir(script_dir)
    try:
        runpy.run_path(script_path, run_name="__main__")
    finally:
        sys.argv = old_argv
        sys.path[:] = old_path
        os.chdir(old_cwd)

## Train one epoch

`--num-train-epochs 1` lets the trainer compute the exact step count for one full pass over
`train`, regardless of effective batch size -- so this is correct whether Kaggle gives you one
GPU or two (DataParallel silently doubles the effective batch size across GPUs).

In [ ]:
run_cli(f"{NLP_DIR}/translation/train.py", [
    "--model", BASE_MODEL,
    "--num-train-epochs", "1",
    "--per-device-train-batch-size", "16",
    "--warmup-steps", "300",
    "--save-steps", "500",
    "--eval-steps", "500",
    "--logging-steps", "50",
    "--disable-tqdm",
    "--push-to-hub",
    "--hub-repo-id", HUB_REPO_ID,
    "--tag", TAG,
])

## Eval this epoch's model on the full test split

Scores the `TAG` revision just pushed (pinned by tag, not just "whatever's on main right
now") against the fixed `test` split -- the same benchmark every other run is scored against,
so these numbers are directly comparable across epochs and against the
`facebook/nllb-200-distilled-600M` baseline.

In [ ]:
run_cli(f"{EVAL_DIR}/eval_mt.py", [
    "--model", HUB_REPO_ID,
    "--revision", TAG,
    "--results-path", f"mt_results_{TAG}.jsonl",
])

In [ ]:
import json

with open(f"{EVAL_DIR}/mt_results_{TAG}.jsonl") as f:
    lines = f.readlines()
print(f"--- {TAG} ({HUB_REPO_ID}, revision={TAG}) ---")
print(json.dumps(json.loads(lines[-1]), indent=2, ensure_ascii=False))

## Publish this epoch's full eval breakdown onto the model card

Writes the per-direction/per-source BLEU/chrF++ table onto
`gmezan/nllb-quechua-spanish`'s README.md, under a `TAG`-labeled block -- so after all
three sessions, the model card documents every epoch's full results side by side, not just
whatever the latest push happened to be.

In [ ]:
run_cli(f"{EVAL_DIR}/publish_model_card.py", [
    "--model", HUB_REPO_ID,
    "--label", TAG,
    "--results-path", f"{EVAL_DIR}/mt_results_{TAG}.jsonl",
])

## Next session

Change `EPOCH_NUM` to `2` at the top of this notebook and re-run everything -- it will
automatically pick up the `epoch-1` state (via `main`) as the base model, train one more
epoch, push to the same `gmezan/nllb-quechua-spanish` repo, and tag the result `epoch-2`.
Repeat with `EPOCH_NUM = 3` for the third session.

`mt_results_epoch-{1,2,3}.jsonl` (and the `epoch-1`/`epoch-2`/`epoch-3` tags on
`gmezan/nllb-quechua-spanish`) let you compare all three epochs' BLEU/chrF++ side by side
once all three sessions are done -- e.g. `eval_mt.py --model gmezan/nllb-quechua-spanish
--revision epoch-2` always scores exactly that epoch, even after `epoch-3` has been pushed
to `main`.